In [1]:
import os
import numpy as np
import xarray as xr
import capytaine as cpt
from capytaine.io.legacy import export_hydrostatics

In [2]:
print(cpt.__version__)

2.3.1


In [3]:
def run_sphere(x,y):
    input_data_dir = os.getcwd()
    output_dir = os.path.join(input_data_dir, f'outputs_x{str(x)}y{str(y)}')
    os.makedirs(output_dir, exist_ok=True)

    mesh_file = os.path.join(input_data_dir, "sphere.dat")
    mesh = cpt.load_mesh(mesh_file, file_format="nemoh")
    mesh = mesh.translate_x(x)
    mesh = mesh.translate_y(y)
    # Symmetry defined in header of "sphere.dat" is used.
    body = cpt.FloatingBody(
        mesh=mesh,
        lid_mesh=mesh.generate_lid(z=-0.05),
        dofs=cpt.rigid_body_dofs(rotation_center=(x, y, -2.0)),
        center_of_mass=(x, y, -2.0),
        name="floating_sphere"
    )

    body.inertia_matrix = body.compute_rigid_body_inertia()
    body.hydrostatic_stiffness = body.immersed_part().compute_hydrostatic_stiffness()

    #body.show()  # Uncomment to display the mesh in 3D for verification
    #body.show_matplotlib()

    test_matrix = xr.Dataset(coords={
        "omega": np.linspace(0.01, 8.4, 20),
        "radiating_dof": list(body.dofs),
        "wave_direction": [0],
        "water_depth": [50.0],
        "rho": [1000.0],
        })

    solver = cpt.BEMSolver()
    dataset = solver.fill_dataset(test_matrix, body.immersed_part(), n_jobs=2)

    cpt.export_dataset(os.path.join(output_dir, f'sphere_x{str(x)}y{str(y)}.nc'), dataset)
    export_hydrostatics(output_dir, body)

    return dataset
    


In [4]:
# Create the grid of x and y coordinates
#x_values = np.arange(-2, 2.5, 0.5)  # 2.5 to include 2 in the range
#y_values = np.arange(-2, 2.5, 0.5)

# Create a meshgrid
#X, Y = np.meshgrid(x_values, y_values)

# Flatten the arrays to iterate through each combination
#x_flat = X.flatten()
#y_flat = Y.flatten()

# Iterate through each combination of x and y
#for x, y in zip(x_flat, y_flat):
#    run_sphere(x,y)

bemData = run_sphere(2.5,0)


[15:12:10] WARNING  Mesh resolution for 42 problems:                                                               
                    The resolution of the mesh might be insufficient for omega ranging from 6.192 to 8.400.        
                    This warning appears when the largest panel of this mesh has radius > wavelength/8.

Output()

[15:16:07] WARNING  Exporting problem in already existing directory:                                               
                    c:\Users\jtgrasb\Documents\GitHub\WEC-Sim_Applications\Variable_Hydro\largeDispValidation\hydro
                    Data\outputs_x2.5y0                                                                            
                                 You might be overwriting existing files!

In [5]:
print(bemData['added_mass'][0:10,0,0].values)
print(bemData['radiation_damping'][0:10,0,0].values)
print(bemData['hydrostatic_stiffness'])

[            nan 139418.73196236 165207.14757382 162230.17267256
  88471.9223701   51558.81148857  43365.56055161  44647.66199968
  48360.76402244  52222.06273198]
[            nan    206.0421409   13910.72669444 113871.52506506
 185343.80585448 162085.14156905 123195.93411789  90768.79075331
  66855.98119575  49847.30808465]
<xarray.DataArray 'hydrostatic_stiffness' (influenced_dof: 6, radiating_dof: 6)> Size: 288B
array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  7.69965687e+05,
        -0.00000000e+00, -5.44564394e-12,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00, -0.00000000e+00,
         5.12984251e+06, -2.17825757e-12, -3.49834561e-05],
       [ 0.00000000e+00,  0.00000000e+00, -5.44564394e-12,
        -2.17825757e-12,  5.12984251e+06, -6.